<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_10-2_Multi_Agent_%EC%97%AC%ED%96%89%EA%B3%84%ED%9A%8D_%EB%8F%84%EC%9A%B0%EB%AF%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 2. Multi-Agent 기반 여행 계획 도우미

## 학습 목표
1. 하나의 목표를 **서로 독립적인 하위 과제**로 나누는 방법을 설명한다.
2. `asyncio.gather()`로 **Worker 3개를 병렬 실행**한다.
3. Worker가 반환한 구조화된 결과(JSON과 호환되는 딕셔너리)를 Orchestrator가 검증·통합한다.
4. **순차 실행과 병렬 실행 시간**을 비교하고 병렬화의 한계를 설명한다.

> **정확한 실습 범위:** API 키가 필요 없는 **Multi-Agent 구조 모의 구현**입니다. Worker는 LLM이 아니라 Python 함수이며, 데이터는 노트북이 만드는 **가상 실습용 자료**입니다. 실제 관광지 운영시간·영업 여부·교통편·요금은 확인하지 않습니다. 실행 지연은 `asyncio.sleep()`으로 인위적으로 넣었으므로, 성능 비교는 **동시 실행 원리의 시연**이지 실제 LLM/API 성능 측정이 아닙니다.

**예제 사용자 요청:** “친구 3명과 부산 당일치기 여행을 가려고 해. 볼거리, 맛집, 이동 방법을 조사해서 여행 계획을 작성해 줘.”

친구 3명 + 요청자 1명 = **총 4명**으로 설정합니다.

## 1. 실습 준비: 패키지 설치 없이 실행하기

Colab에서 **런타임 → 모두 실행**을 누르면 아래 셀들이 순서대로 실행됩니다. 별도 파일 업로드나 API 키는 필요하지 않습니다.

In [1]:
# asyncio: 비동기 함수를 정의하고 여러 작업을 동시에 실행하는 데 사용합니다.
import asyncio
# json: Python 딕셔너리를 보기 좋은 JSON 문자열로 출력하는 데 사용합니다.
import json
# time: 작업별 소요 시간을 초 단위로 측정하는 데 사용합니다.
import time
# datetime: 언제 어떤 작업을 실행했는지 로그에 남기는 데 사용합니다.
from datetime import datetime
# Path: 실습용 JSON 파일을 노트북 환경에 저장하고 읽기 위해 사용합니다.
from pathlib import Path

# Colab 기본 환경에서 /content에 파일을 만들고, 일반 Jupyter에서는 현재 폴더에 만듭니다.
DATA_PATH = Path('busan_travel_sample.json')

# 사용자 요청을 구조화합니다. 각 Worker는 이 요청과 자신의 자료만 사용합니다.
REQUEST = {
    'city': '부산',
    'trip_type': '당일치기',
    'party_size': 4,        # 요청자 1명 + 친구 3명
    'budget_per_person': 60000,  # 연습용 목표 예산(원): 실제 가격이 아닙니다.
    'interest': '바다 풍경과 산책',
}
print('여행 요청:', REQUEST)

여행 요청: {'city': '부산', 'trip_type': '당일치기', 'party_size': 4, 'budget_per_person': 60000, 'interest': '바다 풍경과 산책'}


## 2. 실습용 데이터 만들기

**실제 웹 검색을 하지 않습니다.** 관광지·음식점·교통 아이디어를 서로 다른 세 목록에 저장합니다. 음식점 이름과 예상 예산 및 이동 시간은 **가상의 예시값**입니다.

In [2]:
# Worker들이 각자 담당할 자료만 읽을 수 있도록 자료를 세 부분으로 나눕니다.
sample_data = {
    'attractions': [
        {'name': '해운대 해수욕장', 'area': '해운대', 'activity': '해변 산책', 'duration_min': 70, 'estimated_cost': 0},
        {'name': '동백섬 산책로', 'area': '해운대', 'activity': '바다 풍경 감상', 'duration_min': 60, 'estimated_cost': 0},
        {'name': '광안리 해수욕장', 'area': '광안리', 'activity': '해변 산책과 사진 촬영', 'duration_min': 70, 'estimated_cost': 0},
    ],
    'restaurants': [
        {'name': '해운대 식당 A (가상)', 'area': '해운대', 'menu': '돼지국밥', 'estimated_cost': 12000},
        {'name': '광안리 식당 B (가상)', 'area': '광안리', 'menu': '밀면', 'estimated_cost': 11000},
    ],
    'transport': [
        {'from': '부산역', 'to': '해운대', 'method': '대중교통', 'estimated_min': 60},
        {'from': '해운대', 'to': '광안리', 'method': '대중교통', 'estimated_min': 40},
        {'from': '광안리', 'to': '부산역', 'method': '대중교통', 'estimated_min': 50},
    ],
}

# ensure_ascii=False를 주면 JSON 파일에서도 한글이 그대로 표시됩니다.
DATA_PATH.write_text(json.dumps(sample_data, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'실습 데이터 파일 생성: {DATA_PATH.resolve()}')
print('관광지:', len(sample_data['attractions']), '개 / 식당:', len(sample_data['restaurants']), '개')

실습 데이터 파일 생성: /content/busan_travel_sample.json
관광지: 3 개 / 식당: 2 개


## 3. Worker 실행 로그(Trace) 준비

실습에서는 **누가 어떤 작업을 시작하고 끝냈는지** 기록합니다. 병렬 실행 시 작업이 겹치는 것을 로그로 확인할 수 있습니다.

In [3]:
# TRACE 리스트에 Worker의 시작/종료 시각을 저장합니다.
TRACE = []

def record_trace(agent: str, event: str) -> None:
    """에이전트 이름과 현재 이벤트를 추적 목록에 추가합니다."""
    # 사람이 읽기 쉬운 시각과 시간 비교에 유용한 고정밀 시계를 함께 기록합니다.
    TRACE.append({
        'agent': agent,
        'event': event,
        'timestamp': datetime.now().strftime('%H:%M:%S.%f')[:-3],
        'elapsed_clock': time.perf_counter(),
    })


## 4. Worker 3개 만들기

각 Worker는 **자기 분야의 자료만** 사용하며, 서로의 결과를 기다리지 않습니다.

- **관광지 Worker:** 볼거리 후보를 추립니다.
- **맛집 Worker:** 식사 후보를 정리합니다.
- **교통 Worker:** 구간별 이동 방법을 정리합니다.

세 함수의 `await asyncio.sleep(...)`은 **외부 도구 응답을 기다리는 상황을 재현한 가짜 지연**입니다.

In [4]:
async def attraction_worker(request: dict) -> dict:
    """관광지 자료만 조사한 것처럼 처리하고 구조화된 결과를 반환합니다."""
    record_trace('관광지 Worker', '시작')
    # 실제 API 요청의 네트워크 대기 상황을 모의합니다. 실제 검색은 하지 않습니다.
    await asyncio.sleep(0.6)
    # 다른 Worker의 데이터는 읽지 않고 관광지 목록만 조회합니다.
    data = json.loads(DATA_PATH.read_text(encoding='utf-8'))['attractions']
    # 사용자의 관심사와 맞는 예시 관광지 2개를 선택합니다.
    selected = [next(item for item in data if item['area'] == area) for area in ('해운대', '광안리')]
    record_trace('관광지 Worker', '종료')
    return {'worker': 'attractions', 'status': 'success', 'items': selected, 'source': DATA_PATH.name}


async def restaurant_worker(request: dict) -> dict:
    """식당 자료만 조회한 후 메뉴와 예시 가격을 반환합니다."""
    record_trace('맛집 Worker', '시작')
    await asyncio.sleep(0.8)  # 실제 API 지연이 아닌 실습용 대기 시간입니다.
    data = json.loads(DATA_PATH.read_text(encoding='utf-8'))['restaurants']
    # 1인당 예산보다 비싼 식사 후보는 제외합니다.
    selected = [item for item in data if item['estimated_cost'] <= request['budget_per_person']]
    record_trace('맛집 Worker', '종료')
    return {'worker': 'restaurants', 'status': 'success', 'items': selected, 'source': DATA_PATH.name}


async def transport_worker(request: dict) -> dict:
    """이동 구간의 교통수단과 예시 소요 시간을 반환합니다."""
    record_trace('교통 Worker', '시작')
    await asyncio.sleep(0.5)  # 인위적 대기 시간입니다.
    data = json.loads(DATA_PATH.read_text(encoding='utf-8'))['transport']
    record_trace('교통 Worker', '종료')
    return {'worker': 'transport', 'status': 'success', 'items': data, 'source': DATA_PATH.name}


## 5. Orchestrator 구현: 작업 분배 → 결과 검증 → 통합

Orchestrator는 세 Worker를 실행한 뒤 **반환값이 빠지거나 잘못된 형식인지 먼저 확인**합니다. 그다음 실제 데이터를 조회했다는 오해가 없도록 결과에 출처와 예시 데이터라는 표시를 남깁니다.

In [5]:
def validate_worker_result(result: dict, expected_worker: str) -> None:
    """Worker 결과의 최소 형식과 출처를 검사합니다."""
    # 결과가 딕셔너리가 아니면 필요한 키를 읽을 수 없으므로 오류를 발생시킵니다.
    if not isinstance(result, dict):
        raise ValueError(f'{expected_worker}: 결과가 dict가 아닙니다.')
    # 작업 종류, 성공 여부, 결과 배열, 출처 표시가 모두 있는지 확인합니다.
    if result.get('worker') != expected_worker or result.get('status') != 'success':
        raise ValueError(f'{expected_worker}: 작업 종류 또는 상태가 올바르지 않습니다.')
    if not isinstance(result.get('items'), list) or not result['items']:
        raise ValueError(f'{expected_worker}: 결과 목록이 비어 있거나 형식이 잘못되었습니다.')
    if result.get('source') != DATA_PATH.name:
        raise ValueError(f'{expected_worker}: 예제 데이터 출처가 일치하지 않습니다.')


def make_travel_plan(request: dict, results: list[dict]) -> dict:
    """검증된 세 Worker 결과를 하나의 여행 계획으로 합칩니다."""
    # 순서가 바뀌어도 Worker 이름으로 결과를 찾도록 사전을 만듭니다.
    by_worker = {result['worker']: result for result in results}
    # 세 Worker의 목록을 가져옵니다. 없는 항목은 앞 단계의 검증에서 차단됩니다.
    attractions = by_worker['attractions']['items']
    restaurants = by_worker['restaurants']['items']
    transports = by_worker['transport']['items']

    # 해운대 관광지와 해운대 식당을 묶어 불필요한 지역 이동을 줄이는 간단한 규칙입니다.
    first_attraction = next(item for item in attractions if item['area'] == '해운대')
    second_attraction = next(item for item in attractions if item['area'] == '광안리')
    lunch = next(item for item in restaurants if item['area'] == '해운대')

    # 일정은 순서만 안내합니다. 아래 시간은 샘플 데이터의 예시 추정치이며 실제 운행표가 아닙니다.
    itinerary = [
        {'step': 1, 'category': '이동', 'description': '부산역 → 해운대', 'minutes': transports[0]['estimated_min']},
        {'step': 2, 'category': '관광', 'description': first_attraction['name'], 'minutes': first_attraction['duration_min']},
        {'step': 3, 'category': '점심', 'description': f"{lunch['name']} · {lunch['menu']}", 'minutes': 60},
        {'step': 4, 'category': '이동', 'description': '해운대 → 광안리', 'minutes': transports[1]['estimated_min']},
        {'step': 5, 'category': '관광', 'description': second_attraction['name'], 'minutes': second_attraction['duration_min']},
        {'step': 6, 'category': '이동', 'description': '광안리 → 부산역', 'minutes': transports[2]['estimated_min']},
    ]
    # 실제 교통 요금·기타 경비가 없으므로 확정 총경비는 계산하지 않습니다.
    known_sample_cost = lunch['estimated_cost'] + first_attraction['estimated_cost'] + second_attraction['estimated_cost']
    return {
        'title': f"{request['city']} {request['trip_type']} 여행 계획 (실습용)",
        'party_size': request['party_size'],
        'itinerary': itinerary,
        'estimated_total_minutes': sum(step['minutes'] for step in itinerary),
        'sample_known_cost_per_person': known_sample_cost,
        'budget_per_person': request['budget_per_person'],
        'missing_costs': ['교통비', '간식비', '기타 개인 경비'],
        'sources': sorted({result['source'] for result in results}),
        'notice': '가상/예시 자료로 만든 계획입니다. 실제 시간·영업 여부·요금은 별도 확인이 필요합니다.',
    }


async def orchestrate_parallel(request: dict) -> dict:
    """여러 Worker를 동시에 실행한 후 결과를 확인하고 통합합니다."""
    record_trace('Orchestrator', '작업 분배')
    # gather는 세 코루틴을 함께 예약하고 모두 완료될 때까지 기다립니다.
    # 반환 순서는 gather에 전달한 함수 순서와 같습니다.
    results = await asyncio.gather(
        attraction_worker(request),
        restaurant_worker(request),
        transport_worker(request),
    )
    # 잘못된 형식의 결과를 최종 답변에 그대로 사용하지 않도록 검증합니다.
    for name, result in zip(('attractions', 'restaurants', 'transport'), results):
        validate_worker_result(result, name)
    record_trace('Orchestrator', '결과 검증 완료')
    plan = make_travel_plan(request, results)
    record_trace('Orchestrator', '최종 통합 완료')
    return plan


## 6. 병렬 실행: 한 번에 세 Worker 호출하기

**예상 관찰:** 각 Worker의 시작 기록이 연달아 나타나고, 대기가 끝나는 순서에 따라 종료됩니다. 가장 오래 기다리는 Worker가 전체 시간에 큰 영향을 줍니다.

In [6]:
# 여러 번 실행하더라도 이전 실행 로그가 섞이지 않도록 목록을 비웁니다.
TRACE.clear()
# 노트북 최상위 셀에서는 await를 직접 사용할 수 있습니다(Colab/Jupyter 지원).
parallel_start = time.perf_counter()
parallel_plan = await orchestrate_parallel(REQUEST)
parallel_seconds = time.perf_counter() - parallel_start

# 한글이 깨지지 않도록 ensure_ascii=False로 JSON 형태의 최종 계획을 보여줍니다.
print(json.dumps(parallel_plan, ensure_ascii=False, indent=2))
print(f'\n병렬 실행 소요 시간(실습용 모의 대기): {parallel_seconds:.3f}초')
print('\n[실행 추적 로그]')
for item in TRACE:
    print(f"{item['timestamp']} | {item['agent']:15s} | {item['event']}")

{
  "title": "부산 당일치기 여행 계획 (실습용)",
  "party_size": 4,
  "itinerary": [
    {
      "step": 1,
      "category": "이동",
      "description": "부산역 → 해운대",
      "minutes": 60
    },
    {
      "step": 2,
      "category": "관광",
      "description": "해운대 해수욕장",
      "minutes": 70
    },
    {
      "step": 3,
      "category": "점심",
      "description": "해운대 식당 A (가상) · 돼지국밥",
      "minutes": 60
    },
    {
      "step": 4,
      "category": "이동",
      "description": "해운대 → 광안리",
      "minutes": 40
    },
    {
      "step": 5,
      "category": "관광",
      "description": "광안리 해수욕장",
      "minutes": 70
    },
    {
      "step": 6,
      "category": "이동",
      "description": "광안리 → 부산역",
      "minutes": 50
    }
  ],
  "estimated_total_minutes": 350,
  "sample_known_cost_per_person": 12000,
  "budget_per_person": 60000,
  "missing_costs": [
    "교통비",
    "간식비",
    "기타 개인 경비"
  ],
  "sources": [
    "busan_travel_sample.json"
  ],
  "notice": "가상/예시 자료로 만든 계획입니다. 실제 시간·영업 여부·요금은

## 7. 사람이 읽기 쉬운 여행 일정표 출력

일정에는 **순서와 예시 소요 시간**을 표시합니다. 실제 이동 시간·장소 영업 여부를 검증한 일정은 아닙니다.

In [7]:
# 저장된 최종 계획을 항목별로 읽어 간단한 일정표로 출력합니다.
print('=', parallel_plan['title'], '=')
print('인원:', parallel_plan['party_size'], '명')
for step in parallel_plan['itinerary']:
    # 각 단계의 작업 유형과 내용을 보여 주어 Orchestrator의 통합 결과를 확인합니다.
    print(f"{step['step']}. [{step['category']}] {step['description']} (예시 {step['minutes']}분)")
print('예시 총 소요 시간:', parallel_plan['estimated_total_minutes'], '분')
print('예시로 확인 가능한 1인 비용:', f"{parallel_plan['sample_known_cost_per_person']:,}", '원')
print('별도 확인 비용:', ', '.join(parallel_plan['missing_costs']))
print('주의:', parallel_plan['notice'])

= 부산 당일치기 여행 계획 (실습용) =
인원: 4 명
1. [이동] 부산역 → 해운대 (예시 60분)
2. [관광] 해운대 해수욕장 (예시 70분)
3. [점심] 해운대 식당 A (가상) · 돼지국밥 (예시 60분)
4. [이동] 해운대 → 광안리 (예시 40분)
5. [관광] 광안리 해수욕장 (예시 70분)
6. [이동] 광안리 → 부산역 (예시 50분)
예시 총 소요 시간: 350 분
예시로 확인 가능한 1인 비용: 12,000 원
별도 확인 비용: 교통비, 간식비, 기타 개인 경비
주의: 가상/예시 자료로 만든 계획입니다. 실제 시간·영업 여부·요금은 별도 확인이 필요합니다.


## 8. 비교 실험: 순차 실행 vs 병렬 실행

**핵심:** 순차 실행은 각 Worker를 하나씩 기다립니다. 병렬 실행은 세 Worker의 기다리는 시간이 겹칩니다.

> 이 실험은 `asyncio.sleep`으로 만든 **대기 시간 비교**입니다. 실제 LLM 비용·토큰 수·응답 품질을 측정하지 않습니다. CPU를 계속 쓰는 무거운 계산에서는 같은 방식으로 빨라진다고 단정할 수 없습니다.

In [8]:
async def orchestrate_sequential(request: dict) -> dict:
    """비교를 위해 Worker를 하나씩 끝날 때까지 기다리는 버전입니다."""
    # await를 각각 실행하므로 앞선 작업이 끝나야 다음 Worker가 시작합니다.
    attraction = await attraction_worker(request)
    restaurant = await restaurant_worker(request)
    transport = await transport_worker(request)
    # 비교의 공정성을 위해 병렬 버전과 똑같이 결과를 검증·통합합니다.
    results = [attraction, restaurant, transport]
    for name, result in zip(('attractions', 'restaurants', 'transport'), results):
        validate_worker_result(result, name)
    return make_travel_plan(request, results)

# 순차 실행 시간을 측정합니다.
sequential_start = time.perf_counter()
sequential_plan = await orchestrate_sequential(REQUEST)
sequential_seconds = time.perf_counter() - sequential_start

# 앞에서 실행한 병렬 버전의 결과와 시간은 그대로 사용합니다.
print(f'순차 실행: {sequential_seconds:.3f}초')
print(f'병렬 실행: {parallel_seconds:.3f}초')
print(f'시간 절약: {sequential_seconds - parallel_seconds:.3f}초')
print(f'속도 비율(순차/병렬): {sequential_seconds / parallel_seconds:.2f}배')

# 두 방식이 동일한 계획을 만드는지 확인하여 기능을 바꾸지 않고 속도만 비교합니다.
assert sequential_plan == parallel_plan, '두 방식의 계획 내용이 다릅니다.'
print('검사 통과: 순차와 병렬의 최종 계획이 같습니다.')

순차 실행: 1.906초
병렬 실행: 0.801초
시간 절약: 1.104초
속도 비율(순차/병렬): 2.38배
검사 통과: 순차와 병렬의 최종 계획이 같습니다.


## 9. 자동 검사

세 Worker의 결과가 정확한 구조를 따르는지, 잘못된 결과가 차단되는지 확인합니다.

In [9]:
# 테스트 1: Worker의 개수에 맞게 최종 계획이 생성되었는지 확인합니다.
assert parallel_plan['party_size'] == 4
assert len(parallel_plan['itinerary']) == 6

# 테스트 2: 결과에 가상 자료 출처와 미확인 비용 경고가 포함되는지 확인합니다.
assert DATA_PATH.name in parallel_plan['sources']
assert '교통비' in parallel_plan['missing_costs']

# 테스트 3: 결과 목록이 비어 있는 잘못된 Worker 응답을 거부하는지 확인합니다.
invalid_result = {'worker': 'attractions', 'status': 'success', 'items': [], 'source': DATA_PATH.name}
try:
    validate_worker_result(invalid_result, 'attractions')
except ValueError:
    print('검사 통과: 비어 있는 Worker 결과를 차단했습니다.')
else:
    raise AssertionError('빈 결과를 차단하지 못했습니다.')

print('모든 자동 검사를 통과했습니다.')

검사 통과: 비어 있는 Worker 결과를 차단했습니다.
모든 자동 검사를 통과했습니다.
